# 04 — LangChain LCEL vs LlamaIndex Pipeline

Side-by-side comparison:
- Latency (wall-clock, p50/p95)
- Answer faithfulness (RAGAS)
- Answer relevancy (RAGAS)
- Implementation lines of code

## Setup
Both pipelines are loaded from the same ingested Weaviate tenant.

In [ ]:
import asyncio, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from production_rag.core.config import get_settings
from production_rag.core.llm_client import get_llm_client
from production_rag.core.logging import setup_logging
from production_rag.ingestion.embedder import get_embedder
from production_rag.vectorstore.weaviate_client import get_weaviate_client
from production_rag.chains.rag_chain import RAGChain
from production_rag.chains.llamaindex_pipeline import LlamaIndexPipeline

setup_logging(json_logs=False)
settings = get_settings()
weaviate = get_weaviate_client(settings)
embedder = get_embedder(settings)
llm = get_llm_client(settings)

rag_chain = RAGChain(weaviate, embedder, llm, settings)
print("RAGChain ready")

In [ ]:
TENANT_ID = "default"
QUERIES = [
    "What are the main contributions of the RAG paper?",
    "How does RAPTOR handle hierarchical document summarisation?",
    "What is reciprocal rank fusion?",
    "Explain the difference between dense and sparse retrieval.",
    "How does CRAG improve retrieval quality?",
]

async def benchmark_lcel(queries):
    results = []
    for q in queries:
        t0 = time.perf_counter()
        resp = await rag_chain.invoke(q, TENANT_ID, enable_crag=True, enable_self_rag=False)
        latency = (time.perf_counter() - t0) * 1000
        results.append({"query": q, "answer": resp.answer, "latency_ms": latency})
    return results

lcel_results = asyncio.run(benchmark_lcel(QUERIES))
pd.DataFrame(lcel_results)[["query", "latency_ms"]]

In [ ]:
# LlamaIndex pipeline requires indexed documents
# Skipped here if LlamaIndex is not installed — compare outputs manually
try:
    from llama_index.core import SimpleDirectoryReader
    # li_pipeline = LlamaIndexPipeline(llm, settings)
    # li_pipeline.build(docs)  # provide docs from loader
    print("LlamaIndex available — run build() with your documents")
except ImportError:
    print("LlamaIndex not installed. Install with: pip install llama-index")

In [ ]:
# Latency distribution for LCEL chain
latencies = [r["latency_ms"] for r in lcel_results]
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(range(len(latencies)), latencies, color="steelblue")
ax.set_yticks(range(len(QUERIES)))
ax.set_yticklabels([q[:50] for q in QUERIES])
ax.axvline(np.median(latencies), color="red", linestyle="--", label=f"Median: {np.median(latencies):.0f}ms")
ax.set_xlabel("Latency (ms)")
ax.set_title("LCEL RAGChain — per-query latency")
ax.legend()
plt.tight_layout()
plt.show()